# 01 — Analyse exploratoire (EDA)

Exploration du dataset Credit Card Fraud (ou synthétique).
Objectifs :
- Comprendre la distribution des features
- Visualiser le déséquilibre des classes
- Identifier les features les plus discriminantes
- Détecter les valeurs aberrantes et valeurs manquantes

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Chargement des données

In [ ]:
from anomaly_detection.data.loader import load_csv, generate_synthetic

DATA_PATH = Path('../data/raw/creditcard.csv')
TARGET_COL = 'Class'

if DATA_PATH.exists():
    df, y = load_csv(DATA_PATH, TARGET_COL)
    print(f'Dataset réel chargé : {len(df):,} lignes')
else:
    print('Dataset Kaggle non trouvé → génération synthétique')
    df, y = generate_synthetic(n_normal=5000, n_anomaly=100, save=False)
    TARGET_COL = 'label'

print(f'Shape : {df.shape}')
print(f'Anomalies : {y.sum():,} ({y.mean()*100:.3f}%)')

## 2. Vue d'ensemble

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
print('Valeurs manquantes :', missing[missing > 0].to_dict() or 'Aucune')

## 3. Déséquilibre des classes

In [ ]:
counts = y.value_counts()
fig = px.bar(
    x=['Normal', 'Anomalie'],
    y=counts.values,
    color=['Normal', 'Anomalie'],
    color_discrete_map={'Normal': '#1D9E75', 'Anomalie': '#D85A30'},
    title='Distribution des classes',
    labels={'x': 'Classe', 'y': 'Nombre d\'échantillons'},
    text=counts.values,
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, height=400)
fig.show()

## 4. Distribution des features par classe

In [ ]:
# Sélectionner les features numériques
num_cols = df.select_dtypes(include='number').columns.tolist()[:10]  # 10 premières

df_plot = df[num_cols].copy()
df_plot['label'] = y.values

fig = make_subplots(rows=2, cols=5, subplot_titles=num_cols)

for i, col in enumerate(num_cols):
    row, col_idx = divmod(i, 5)
    for label, color, name in [(0, '#1D9E75', 'Normal'), (1, '#D85A30', 'Anomalie')]:
        subset = df_plot[df_plot['label'] == label][col]
        fig.add_trace(
            go.Histogram(x=subset, name=name, marker_color=color,
                         opacity=0.6, showlegend=(i == 0)),
            row=row + 1, col=col_idx + 1
        )

fig.update_layout(barmode='overlay', height=500, title='Distribution des features — Normal vs Anomalie')
fig.show()

## 5. Corrélations et features discriminantes

In [ ]:
# Différence de moyenne entre classes (proxy de pouvoir discriminant)
df_full = df.select_dtypes(include='number').copy()
df_full['label'] = y.values

means_normal  = df_full[df_full['label'] == 0].drop('label', axis=1).mean()
means_anomaly = df_full[df_full['label'] == 1].drop('label', axis=1).mean()
diff = (means_anomaly - means_normal).abs().sort_values(ascending=False)

fig = px.bar(
    x=diff.index, y=diff.values,
    title='|Δ moyenne| entre classes — features les plus discriminantes',
    labels={'x': 'Feature', 'y': '|Δ moyenne|'},
    color=diff.values,
    color_continuous_scale='Teal',
)
fig.update_layout(coloraxis_showscale=False, height=400)
fig.show()

print('Top 5 features discriminantes :')
print(diff.head())

## 6. Matrice de corrélation

In [ ]:
corr = df.select_dtypes(include='number').iloc[:, :15].corr()

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Matrice de corrélation (15 premières features)',
    height=550,
)
fig.show()

## 7. Conclusions EDA

- **Déséquilibre** : le dataset est très déséquilibré → justifie l'approche non supervisée
- **Features discriminantes** : certaines features ont des distributions très différentes entre classes
- **Pas de valeurs manquantes** : pas de preprocessing d'imputation nécessaire
- **Normalisation recommandée** : `RobustScaler` pour gérer les valeurs extrêmes dans `Amount`